In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
import joblib
import os
import json
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from datetime import datetime
from dotenv import load_dotenv

s:\Projects\Predictive Maintenance\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()
mlflow.login()

False

In [4]:
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000"))

# Set an experiment name (creates it if it doesn't exist)
experiment_name = "predictive-maintenance"
mlflow.set_experiment(experiment_name)
print(f"MLflow Version: {mlflow.__version__}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

2026/04/13 22:31:16 INFO mlflow.tracking.fluent: Experiment with name 'predictive-maintenance' does not exist. Creating a new experiment.


MLflow Version: 3.11.1
Tracking URI: http://localhost:5000


In [6]:
df = pd.read_csv('data/ai4i2020.csv')
df.head(10)

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0
5,6,M14865,M,298.1,308.6,1425,41.9,11,0,0,0,0,0,0
6,7,L47186,L,298.1,308.6,1558,42.4,14,0,0,0,0,0,0
7,8,L47187,L,298.1,308.6,1527,40.2,16,0,0,0,0,0,0
8,9,M14868,M,298.3,308.7,1667,28.6,18,0,0,0,0,0,0
9,10,M14869,M,298.5,309.0,1741,28.0,21,0,0,0,0,0,0


In [14]:
failure_columns = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
print(df.columns.tolist())

['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure']


In [15]:
df.columns = df.columns.str.strip()

# Define correctly
failure_columns = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# Drop safely
df = df.drop(columns=failure_columns + ['UDI', 'Product ID'], errors='ignore')

# Encode
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Type'] = le.fit_transform(df['Type'])

# Split
X = df.drop('Machine failure', axis=1)
y = df['Machine failure']

In [17]:
X

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,2,298.1,308.6,1551,42.8,0
1,1,298.2,308.7,1408,46.3,3
2,1,298.1,308.5,1498,49.4,5
3,1,298.2,308.6,1433,39.5,7
4,1,298.2,308.7,1408,40.0,9
...,...,...,...,...,...,...
9995,2,298.8,308.4,1604,29.5,14
9996,0,298.9,308.4,1632,31.8,17
9997,2,299.0,308.6,1645,33.4,22
9998,0,299.0,308.7,1408,48.5,25


In [21]:
y
y.value_counts()

Machine failure
0    9652
1     348
Name: count, dtype: int64

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [23]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Save feature names
feature_names = X.columns.tolist()

# ------------------------------
# 2. Train Model & Log to MLflow
# ------------------------------

In [24]:
# Define parameters
params = {
    "n_estimators": 100,
    "max_depth": 10,
    "random_state": 42,
    "class_weight": "balanced"
}

In [25]:
# Start an MLflow run
with mlflow.start_run(run_name=f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
    # Log parameters
    mlflow.log_params(params)
    
    # Train model
    model = RandomForestClassifier(**params)
    model.fit(X_train_scaled, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred)
    }
    
    # Log metrics
    mlflow.log_metrics(metrics)
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    
    # Log model with signature and input example
    signature = infer_signature(X_train_scaled, model.predict(X_train_scaled))
    input_example = X_train_scaled[:5]
    
    # Log the model
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        registered_model_name="predictive-maintenance-model" # This registers it!
    )
    
    # Log preprocessing objects as artifacts
    os.makedirs("temp_artifacts", exist_ok=True)
    joblib.dump(scaler, "temp_artifacts/scaler.pkl")
    joblib.dump(le, "temp_artifacts/label_encoder.pkl")
    with open("temp_artifacts/feature_names.txt", "w") as f:
        f.write(','.join(feature_names))
    
    mlflow.log_artifacts("temp_artifacts", artifact_path="preprocessing")
    
    # Clean up temp artifacts
    import shutil
    shutil.rmtree("temp_artifacts")
    
    print(f"Run finished. Metrics: {metrics}")

2026/04/13 22:44:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 22:44:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1930
           1       0.71      0.66      0.68        70

    accuracy                           0.98      2000
   macro avg       0.85      0.82      0.84      2000
weighted avg       0.98      0.98      0.98      2000



Successfully registered model 'predictive-maintenance-model'.
2026/04/13 22:44:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: predictive-maintenance-model, version 1
Created version '1' of model 'predictive-maintenance-model'.


Run finished. Metrics: {'accuracy': 0.9785, 'precision': 0.7076923076923077, 'recall': 0.6571428571428571, 'f1_score': 0.6814814814814815}
🏃 View run run_20260413_224404 at: http://localhost:5000/#/experiments/3/runs/d1c57f9180b5417eb44b4658a31b7c2a
🧪 View experiment at: http://localhost:5000/#/experiments/3
